In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

In [3]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\maria\Desktop\EEG-special-course")
PROCESSED_ROOT = PROJECT_ROOT / "data_processed"

features_file = PROCESSED_ROOT / "features_pre_ses1_with_posterior.csv"

features = pd.read_csv(features_file)

display(features.head())

,participant_id,age,condition,n_occipital_channels,n_posterior_channels,posterior_channels,n_epochs_kept,alpha_power_8_12,posterior_alpha_sum_8_12,fooof_alpha_cf,fooof_alpha_pw,fooof_alpha_bw,fooof_r2,fooof_error
0,sub-001,60,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",77,-10.257854,-9.067168,10.109051,1.711548,1.850375,0.958282,0.085840
1,sub-001,60,EyesOpen,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",35,-11.266725,-9.949122,10.994871,0.382865,3.216583,0.973884,0.050167
2,sub-002,67,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",81,-10.221930,-9.088553,9.204659,1.745596,2.435050,0.966918,0.075781
3,sub-002,67,EyesOpen,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",47,-11.493045,-10.101173,8.467325,0.338656,1.327376,0.977051,0.043199
4,sub-003,44,EyesClosed,3,19,"P7,P3,Pz,P4,P8,PO9,O1,Oz,O2,PO10,P5,P1,P2,P6,P...",107,-10.989451,-9.893156,11.352306,1.386752,1.756418,0.967573,0.067330


In [4]:
def run_paired_rule_baseline(features, feature_col, model_name):
    wide = (
        features
        .pivot_table(
            index="participant_id",
            columns="condition",
            values=feature_col
        )
        .reset_index()
    )

    wide = wide.dropna(subset=["EyesClosed", "EyesOpen"]).copy()

    wide["diff_ec_minus_eo"] = wide["EyesClosed"] - wide["EyesOpen"]

    # Rule: condition with higher feature value is classified as EC
    wide["correct"] = wide["diff_ec_minus_eo"] > 0

    accuracy = wide["correct"].mean()

    return {
        "model": model_name,
        "feature_set": feature_col,
        "model_type": "paired rule baseline",
        "accuracy": accuracy,
        "n_participants": len(wide),
        "n_correct": wide["correct"].sum(),
        "n_incorrect": (~wide["correct"]).sum()
    }

In [5]:
paired_rule_results = []

paired_rule_results.append(
    run_paired_rule_baseline(
        features,
        feature_col="alpha_power_8_12",
        model_name="paired_occipital_alpha_rule"
    )
)

paired_rule_results.append(
    run_paired_rule_baseline(
        features,
        feature_col="posterior_alpha_sum_8_12",
        model_name="paired_posterior_alpha_rule"
    )
)

paired_rule_results.append(
    run_paired_rule_baseline(
        features,
        feature_col="fooof_alpha_pw",
        model_name="paired_fooof_alpha_power_rule"
    )
)

paired_rule_results_df = pd.DataFrame(paired_rule_results)

display(paired_rule_results_df)

,model,feature_set,model_type,accuracy,n_participants,n_correct,n_incorrect
0,paired_occipital_alpha_rule,alpha_power_8_12,paired rule baseline,0.959322,590,566,24
1,paired_posterior_alpha_rule,posterior_alpha_sum_8_12,paired rule baseline,0.957627,590,565,25
2,paired_fooof_alpha_power_rule,fooof_alpha_pw,paired rule baseline,0.973485,528,514,14


In [6]:
def make_paired_difference_table(features, feature_cols):
    paired = None

    for feature in feature_cols:
        wide = (
            features
            .pivot_table(
                index="participant_id",
                columns="condition",
                values=feature
            )
            .reset_index()
        )

        wide = wide.dropna(subset=["EyesClosed", "EyesOpen"]).copy()

        wide[f"{feature}_diff_ec_minus_eo"] = (
            wide["EyesClosed"] - wide["EyesOpen"]
        )

        keep = wide[["participant_id", f"{feature}_diff_ec_minus_eo"]]

        if paired is None:
            paired = keep
        else:
            paired = paired.merge(keep, on="participant_id", how="inner")

    return paired

In [7]:
paired_feature_cols = [
    "alpha_power_8_12",
    "posterior_alpha_sum_8_12",
    "fooof_alpha_cf",
    "fooof_alpha_pw",
    "fooof_alpha_bw"
]

paired_df = make_paired_difference_table(
    features,
    feature_cols=paired_feature_cols
)

display(paired_df.head())
print("Number of participants:", len(paired_df))

condition,participant_id,alpha_power_8_12_diff_ec_minus_eo,posterior_alpha_sum_8_12_diff_ec_minus_eo,fooof_alpha_cf_diff_ec_minus_eo,fooof_alpha_pw_diff_ec_minus_eo,fooof_alpha_bw_diff_ec_minus_eo
0,sub-001,1.008871,0.881954,-0.885820,1.328683,-1.366208
1,sub-002,1.271115,1.012621,0.737334,1.406940,1.107674
2,sub-003,0.934561,0.791942,-0.178562,1.143584,0.371027
3,sub-004,1.593936,1.558882,0.057395,2.084942,0.334469
4,sub-005,0.765474,0.887320,-0.122099,0.797447,0.680985


Number of participants: 528


In [8]:
def make_pairwise_classification_data(paired_df):
    diff_cols = [col for col in paired_df.columns if col.endswith("_diff_ec_minus_eo")]

    rows = []

    for _, row in paired_df.iterrows():
        participant_id = row["participant_id"]

        diffs = row[diff_cols].values.astype(float)

        # EC - EO: first condition is EC
        rows.append({
            "participant_id": participant_id,
            "label": 1,
            **{col: value for col, value in zip(diff_cols, diffs)}
        })

        # EO - EC: first condition is EO
        rows.append({
            "participant_id": participant_id,
            "label": 0,
            **{col: -value for col, value in zip(diff_cols, diffs)}
        })

    pairwise_df = pd.DataFrame(rows)

    return pairwise_df

In [9]:
pairwise_df = make_pairwise_classification_data(paired_df)

display(pairwise_df.head())
print(pairwise_df["label"].value_counts())

,participant_id,label,alpha_power_8_12_diff_ec_minus_eo,posterior_alpha_sum_8_12_diff_ec_minus_eo,fooof_alpha_cf_diff_ec_minus_eo,fooof_alpha_pw_diff_ec_minus_eo,fooof_alpha_bw_diff_ec_minus_eo
0,sub-001,1,1.008871,0.881954,-0.885820,1.328683,-1.366208
1,sub-001,0,-1.008871,-0.881954,0.885820,-1.328683,1.366208
2,sub-002,1,1.271115,1.012621,0.737334,1.406940,1.107674
3,sub-002,0,-1.271115,-1.012621,-0.737334,-1.406940,-1.107674
4,sub-003,1,0.934561,0.791942,-0.178562,1.143584,0.371027


label
1    528
0    528
Name: count, dtype: int64


In [10]:
def run_paired_logistic_model(pairwise_df, feature_cols, model_name):
    data = pairwise_df.dropna(subset=feature_cols + ["label"]).copy()

    X = data[feature_cols].values
    y = data["label"].values
    groups = data["participant_id"].values

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    cv = GroupKFold(n_splits=5)

    y_pred = cross_val_predict(
        model,
        X,
        y,
        groups=groups,
        cv=cv,
        method="predict"
    )

    y_proba = cross_val_predict(
        model,
        X,
        y,
        groups=groups,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    accuracy = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)

    return {
        "model": model_name,
        "model_type": "paired logistic regression",
        "features": ", ".join(feature_cols),
        "accuracy": accuracy,
        "auc": auc,
        "n_samples": len(data),
        "n_participants": data["participant_id"].nunique()
    }

In [11]:
diff_alpha = ["alpha_power_8_12_diff_ec_minus_eo"]

diff_posterior = ["posterior_alpha_sum_8_12_diff_ec_minus_eo"]

diff_fooof = [
    "fooof_alpha_cf_diff_ec_minus_eo",
    "fooof_alpha_pw_diff_ec_minus_eo",
    "fooof_alpha_bw_diff_ec_minus_eo"
]

diff_combined = [
    "alpha_power_8_12_diff_ec_minus_eo",
    "posterior_alpha_sum_8_12_diff_ec_minus_eo",
    "fooof_alpha_cf_diff_ec_minus_eo",
    "fooof_alpha_pw_diff_ec_minus_eo",
    "fooof_alpha_bw_diff_ec_minus_eo"
]

paired_logistic_results = []

paired_logistic_results.append(
    run_paired_logistic_model(
        pairwise_df,
        feature_cols=diff_alpha,
        model_name="paired_logreg_occipital_alpha"
    )
)

paired_logistic_results.append(
    run_paired_logistic_model(
        pairwise_df,
        feature_cols=diff_posterior,
        model_name="paired_logreg_posterior_alpha"
    )
)

paired_logistic_results.append(
    run_paired_logistic_model(
        pairwise_df,
        feature_cols=diff_fooof,
        model_name="paired_logreg_fooof"
    )
)

paired_logistic_results.append(
    run_paired_logistic_model(
        pairwise_df,
        feature_cols=diff_combined,
        model_name="paired_logreg_combined"
    )
)

paired_logistic_results_df = pd.DataFrame(paired_logistic_results)

display(paired_logistic_results_df)

,model,model_type,features,accuracy,auc,n_samples,n_participants
0,paired_logreg_occipital_alpha,paired logistic regression,alpha_power_8_12_diff_ec_minus_eo,0.973485,0.996377,1056,528
1,paired_logreg_posterior_alpha,paired logistic regression,posterior_alpha_sum_8_12_diff_ec_minus_eo,0.969697,0.996506,1056,528
2,paired_logreg_fooof,paired logistic regression,"fooof_alpha_cf_diff_ec_minus_eo, fooof_alpha_p...",0.967803,0.996940,1056,528
3,paired_logreg_combined,paired logistic regression,"alpha_power_8_12_diff_ec_minus_eo, posterior_a...",0.973485,0.997719,1056,528


In [12]:
comparison = pd.concat(
    [
        paired_rule_results_df,
        paired_logistic_results_df
    ],
    ignore_index=True,
    sort=False
)

display(comparison)

,model,feature_set,model_type,accuracy,n_participants,n_correct,n_incorrect,features,auc,n_samples
0,paired_occipital_alpha_rule,alpha_power_8_12,paired rule baseline,0.959322,590,566.0,24.0,NaN,NaN,NaN
1,paired_posterior_alpha_rule,posterior_alpha_sum_8_12,paired rule baseline,0.957627,590,565.0,25.0,NaN,NaN,NaN
2,paired_fooof_alpha_power_rule,fooof_alpha_pw,paired rule baseline,0.973485,528,514.0,14.0,NaN,NaN,NaN
3,paired_logreg_occipital_alpha,NaN,paired logistic regression,0.973485,528,NaN,NaN,alpha_power_8_12_diff_ec_minus_eo,0.996377,1056.0
4,paired_logreg_posterior_alpha,NaN,paired logistic regression,0.969697,528,NaN,NaN,posterior_alpha_sum_8_12_diff_ec_minus_eo,0.996506,1056.0
5,paired_logreg_fooof,NaN,paired logistic regression,0.967803,528,NaN,NaN,"fooof_alpha_cf_diff_ec_minus_eo, fooof_alpha_p...",0.996940,1056.0
6,paired_logreg_combined,NaN,paired logistic regression,0.973485,528,NaN,NaN,"alpha_power_8_12_diff_ec_minus_eo, posterior_a...",0.997719,1056.0
